I will be going over ensemble methods for trees. These are ways to create a better tree-based model because a single decision tree has high variance since it will create splits until there is no rss error or the leaves are completely pure. So, in short we are going to create a bunch of decision trees and use them to get 1 prediction per instance. 

First I will go over bagging, this is a method of creating N different decision trees using bootstrap resampling on the training data to get different trees and then aggregating their predictions to get one final one. So, you choose how many trees to create and then we use bootstrap resampling to get N bagged datasets to train each tree on, we allow each to grow as deep as it can (high variance). Then, we are going to feed the model an instance and it passes it into each of the N trees and then if it is regression we will get the prediciton of each tree and take the avg of those predictions (aggregating) and if classification we will use voting to keep track of how many times each class was predicted and then predict the highest proprtion class. This is going to cancel out the variance that each tree exhibits because it is taking the avg, so each tree is pulling in a different direction and the avg gets the middle. 

There are also voting classifiers which create an ensemble of DIFFERENT models that are going to each train on the same data, but the prediction of this model is going to be the class that has the highest proporiton amoung the ensemble. This is going to bring our accuracy up because if we use models that are of different structure(logistic regression vs decision trees, line vs box) each error will be different so then we can cancel out the errors in the far directions and choose the middle of them, because the class that most models agree with is most likely the right class. Sklearn allows us to do this using the VotingClassifer class

In [1]:
#Voting Classifier predictions-- going to use classification tree, logistic regression, svm
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [2]:
X,y = make_moons(n_samples=500, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
#now create the voting classifier ensemble method
voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)),
        ('tree', DecisionTreeClassifier(random_state=42)),
        ('svm', SVC(random_state=42))
    ]
)
voting_clf.fit(X_train,y_train)

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('lr', ...), ('tree', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'hard'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels.","ndarray[int64](2,)","[0,1]"
"estimators_ estimators_: list of classifiersThe collection of fitted sub-estimators as defined in ``estimators``that are not 'drop'. Note that sub-estimators are always fitted oninteger-encoded labels (see ``le_`` attribute). When ``y`` containsnon-integer class labels (e.g. strings), use ``le_.inverse_transform``to map predictions back to the original label space.",list,"[LogisticRegre...ndom_state=42), DecisionTreeC...ndom_state=42), SVC(random_state=42)]"
le_ le_: :class:`~sklearn.preprocessing.LabelEncoder`Transformer used to encode the labels during fit and decode duringprediction. Sub-estimators in ``estimators_`` are fitted on theinteger-encoded labels produced by this encoder.,LabelEncoder,LabelEncoder()
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying classifier exposes such an attribute when fit... versionadded:: 0.24,int,2


In [3]:
#now lets get the accuracy of each individual estimator in the voting classifier
for name, model in voting_clf.named_estimators_.items():
    print(f'{name} = {model.score(X_test, y_test)}')


lr = 0.85
tree = 0.82
svm = 0.87


In [4]:
#now lets compare to the voting classifier ensemble as a whole
voting_clf.score(X_test, y_test)

0.85

Performing bagging using the BaggingClassifer class in sklearn from ensembles. It will allow us to take a bunch of bootstrapped resamples from the data and create N estimators. Then for predictions we are going to get the highest class amount for classification amoung the estimators or the mean for regression

In [5]:
from sklearn.ensemble import BaggingClassifier

#we are going to be creating 500 different classification trees, each will be training from 100 bootstrapped samples of the orignal data
#and we are using n_jobs = -1 so we are using all the available cores to make it faster to train all of them
bag_clf = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, max_samples=100, n_jobs=-1, random_state=42)
bag_clf.fit(X_train, y_train)
bag_clf.score(X_test, y_test) #got 90% test accuracy

0.9

With bagging there is about 37% of the original data that each tree will not see just due to bootstrapping, these are called the out of bag samples. So, we can use these OOB samples in each tree to test them and then get an OOB score back to show how accurate it was

In [6]:
bag_clf2 = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, max_samples=100, n_jobs=-1, random_state=42, oob_score=True)
bag_clf2.fit(X_train, y_train)
bag_clf2.oob_score_ #got 91.5% oob score so we should be around there on the testing set since the model had not seen any of the oob samples before

0.915

In [7]:
bag_clf2.oob_decision_function_ #this gives us the probability of being in each class for each instance we feed the model

array([[0.98670213, 0.01329787],
       [0.01066667, 0.98933333],
       [0.08974359, 0.91025641],
       [0.94670051, 0.05329949],
       [0.95886889, 0.04113111],
       [0.98994975, 0.01005025],
       [0.87002653, 0.12997347],
       [0.82479784, 0.17520216],
       [0.31524548, 0.68475452],
       [0.03768844, 0.96231156],
       [0.45454545, 0.54545455],
       [0.86683417, 0.13316583],
       [0.99744246, 0.00255754],
       [0.1425    , 0.8575    ],
       [0.47941889, 0.52058111],
       [0.93175853, 0.06824147],
       [0.97303922, 0.02696078],
       [0.99749373, 0.00250627],
       [0.84130982, 0.15869018],
       [0.2625    , 0.7375    ],
       [0.01578947, 0.98421053],
       [0.01842105, 0.98157895],
       [0.84343434, 0.15656566],
       [0.97022333, 0.02977667],
       [0.74603175, 0.25396825],
       [0.35356201, 0.64643799],
       [0.38560411, 0.61439589],
       [1.        , 0.        ],
       [0.01052632, 0.98947368],
       [0.02682927, 0.97317073],
       [0.

Now I am going to use a random forest classifier for the same dataset. The only difference between the random forest and bagging is that each tree in the random forest is going to choose its splits off of a randomly chosen subset of the number of features rather than all of them. This will speed up training a bit, but more importantly it will decrease any correlation between trees which will cause true independence and difference in variance. This is because in bagging if one of the features dominate the others when choosing a split to minizime the gini or rss score then it will be chosen each time so the trees will be similar. Now, it wont be the same each time which garuntees that there will be variation and this will make the aggregation effective and should draw the prediction of the ensemble method closer to the best purity/rss

In [8]:
from sklearn.ensemble import RandomForestClassifier
#this creates a random forest that will bag 100 samples each time and only decide between square root(number features) each split
forest_clf = RandomForestClassifier(n_estimators=500, criterion='gini', max_features='sqrt',oob_score=True, max_samples=100, random_state=42)
forest_clf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_true, y_pred)` to use acustom metric. Only available if `bootstrap=True`.For an illustration of out-of-bag (OOB) error estimation, see the example:ref:`sphx_glr_auto_examples_ensemble_plot_ensemble_oob.py`.",True
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"max_samples max_samples: int or float, default=NoneIf bootstrap is True, the number of samples to draw from Xto train each base estimator.- If None (default), then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples... versionadded:: 0.22.. versionchanged:: 1.9 Float `max_samples` is relative to `sample_weight.sum()` instead of `X.shape[0]` for weighted samples.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node 

In [9]:
forest_clf.oob_score_ #here we got 92.5% accuracy with the random sampling on the features

0.925

Now I'm going to look at the feature importance of the features in the dataset. This will allow me to see on average how much each feature brought down the overall impurity for each tree in the random forest

In [16]:
forest_clf.feature_importances_ #we can see both features are used pretty closely to reduce the impurity; the first feature reduces impuirty about 
#45% of the time and the 2nd reduces impurity about 54% of the time amoung the trees in the random forest

array([0.4504793, 0.5495207])